# G1 Academy Bonus - Task 7: SLAM operation and map visualization

## Introduction
This task rebuilds the operational SLAM surface described in `notes.txt` section 5, natively: `start_mapping`/`stop_mapping`, `relocate`, `add_point`/`remove_point`, `navigate_to_point`, `navigate_path`, and `view_map`. The deployed RPC operations this academy has verified are mapping start/stop, relocate (`init_pose`), and single-pose navigation - the same IDs `sdk_wrapper._SlamClient` and `slam_util.SlamRpc` use. Native complete-path navigation is version-dependent, so `navigate_path` refuses a silent point-by-point fallback rather than mislabeling it. Once a map is saved on the mainboard it may no longer be accessible for visualization, so `view_map` reads from *snapshots captured while mapping is running*, not from the saved map file afterward.

In [9]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Native `SlamRpc` client + pose parsing
`SlamRpc` is a `unitree_sdk2py.rpc.client.Client` registered against the documented API IDs: `1801` start mapping, `1802`/`1901` stop mapping, `1804` relocate/init-pose, `1102` single-pose navigation. Pose comes back as JSON on `rt/slam_info`/`rt/slam_key_info`; an all-zero pose means "no valid pose yet", not "robot at the origin", so it is treated as absent.

In [10]:
import json
import math
from unitree_sdk2py.idl.nav_msgs.msg.dds_ import Odometry_
from unitree_sdk2py.idl.std_msgs.msg.dds_ import String_
from unitree_sdk2py.rpc.client import Client

class SlamRpc(Client):
    def __init__(self):
        super().__init__("slam_operate", False)
        for api_id in (1801, 1802, 1804, 1102, 1901):
            self._RegistApi(api_id, 0)
        self._SetApiVerson("1.0.0.1")
    def _call_json(self, api_id, payload):
        code, data = self._Call(api_id, json.dumps(payload))
        return int(code), data
    def start_mapping(self, slam_type="indoor"):
        return self._call_json(1801, {"data": {"slam_type": slam_type}})
    def stop_mapping(self, save_path=None):
        if save_path:
            return self._call_json(1802, {"data": {"address": save_path}})
        return self._call_json(1901, {"data": {}})
    def init_pose(self, x, y, yaw, address):
        qz, qw = math.sin(yaw / 2), math.cos(yaw / 2)
        return self._call_json(1804, {"data": {"x": x, "y": y, "z": 0.0, "q_x": 0.0, "q_y": 0.0, "q_z": qz, "q_w": qw, "address": address}})
    def pose_nav(self, x, y, yaw):
        qz, qw = math.sin(yaw / 2), math.cos(yaw / 2)
        return self._call_json(1102, {"data": {"targetPose": {"x": x, "y": y, "z": 0.0, "q_x": 0.0, "q_y": 0.0, "q_z": qz, "q_w": qw}, "mode": 1}})

slam_rpc = SlamRpc()
slam_rpc.SetTimeout(10.0)


In [11]:
slam_info_sub = Latest("rt/slam_info", String_)
slam_key_sub = Latest("rt/slam_key_info", String_)
slam_odom_sub = Latest("rt/unitree/slam_mapping/odom", Odometry_)
_last_slam_pose = None


In [12]:
def _yaw_from_quaternion(qx, qy, qz, qw):
    return math.atan2(2.0 * (qw * qz + qx * qy), 1.0 - 2.0 * (qy * qy + qz * qz))

def _pose_from_mapping(value):
    if isinstance(value, str):
        try:
            return _pose_from_mapping(json.loads(value))
        except Exception:
            return None
    if not isinstance(value, dict):
        return None
    candidate = value.get("data", value)
    for key in ("currentPose", "pose", "targetPose"):
        if isinstance(candidate.get(key), dict):
            found = _pose_from_mapping(candidate[key])
            if found is not None:
                return found
    if all(key in candidate for key in ("x", "y")):
        try:
            x, y = float(candidate["x"]), float(candidate["y"])
            if "yaw" in candidate:
                yaw = float(candidate["yaw"])
            else:
                yaw = _yaw_from_quaternion(
                    float(candidate.get("q_x", 0.0)),
                    float(candidate.get("q_y", 0.0)),
                    float(candidate.get("q_z", 0.0)),
                    float(candidate.get("q_w", 1.0)),
                )
            return (x, y, yaw)
        except Exception:
            return None
    for child in value.values():
        found = _pose_from_mapping(child)
        if found is not None:
            return found
    return None

def _pose_from_odom(msg):
    try:
        pos = msg.pose.pose.position
        ori = msg.pose.pose.orientation
        return (float(pos.x), float(pos.y), _yaw_from_quaternion(float(ori.x), float(ori.y), float(ori.z), float(ori.w)))
    except Exception:
        return None

def current_pose(timeout_s=3.0, allow_zero=False):
    global _last_slam_pose
    deadline = time.time() + max(0.0, float(timeout_s))
    while time.time() < deadline:
        for sub in (slam_info_sub, slam_key_sub):
            if sub.message is None:
                continue
            pose = _pose_from_mapping(sub.message.data)
            if pose is None:
                continue
            if allow_zero or any(abs(value) > 1e-5 for value in pose):
                _last_slam_pose = pose
                return pose
        if slam_odom_sub.message is not None:
            pose = _pose_from_odom(slam_odom_sub.message)
            if pose is not None and (allow_zero or any(abs(value) > 1e-5 for value in pose)):
                _last_slam_pose = pose
                return pose
        time.sleep(0.05)
    return _last_slam_pose


## Task 2 - Mapping lifecycle: `start_mapping` / `stop_mapping` / `relocate`
Start and stop bracket a mapping session; `relocate` requires a fresh pose - navigation must never be trusted before a successful relocate against the saved map.

In [13]:
_map_path = "/home/unitree/test.pcd"

def start_mapping():
    return slam_rpc.start_mapping("indoor")

def stop_mapping(save_path=None):
    return slam_rpc.stop_mapping(save_path or _map_path)

def relocate(pose=None, map_path=None, allow_origin=True):
    if pose is None:
        pose = current_pose(allow_zero=allow_origin)
    if pose is None:
        return {
            "ok": False,
            "error": "No SLAM pose received yet. Start mapping/relocation, or pass pose=(x, y, yaw).",
            "topics": ["rt/slam_info", "rt/slam_key_info", "rt/unitree/slam_mapping/odom"],
        }
    code, raw = slam_rpc.init_pose(*pose, address=map_path or _map_path)
    return {"ok": code == 0, "code": code, "raw": raw, "pose": pose, "map_path": map_path or _map_path}


In [14]:
start_mapping()

(0,
 '{"succeed":true,"errorCode":0,"info":"Successfully started mapping.","data":{}}')

In [15]:
stop_mapping()

(0, '{"succeed":true,"errorCode":0,"info":"Save pcd successfully.","data":{}}')

In [16]:
relocate()
# relocate(pose=(0.0, 0.0, 0.0))  # explicit initial-pose fallback

{'ok': True,
 'code': 0,
 'raw': '{"succeed":true,"errorCode":0,"info":"Successfully started re-location.","data":{}}',
 'pose': (0.0, 0.0, 0.0),
 'map_path': '/home/unitree/test.pcd'}

## Task 3 - `add_point(point_name)` / `remove_point(point_name)`
Named poses are only meaningful once localization is valid, so they are captured from `current_pose()` (the live relocated pose) and persisted to a small JSON file, not kept only in memory.

In [ ]:
import os
from pathlib import Path

_points_path = Path("slam_points.json")
def _load_points():
    return json.loads(_points_path.read_text()) if _points_path.exists() else {}
def _save_points(points):
    tmp = _points_path.with_suffix(".tmp")
    tmp.write_text(json.dumps(points, indent=2))
    os.replace(tmp, _points_path)
_points = _load_points()

def add_point(point_name):
    pose = current_pose()
    if pose is None:
        return {"ok": False, "error": "No valid SLAM pose; relocate first or wait for SLAM odometry."}
    _points[point_name] = pose
    _save_points(_points)
    return {"ok": True, "pose": pose}

def remove_point(point_name):
    _points.pop(point_name, None)
    _save_points(_points)

add_point("pickup")
# remove_point("pickup")

In [18]:
add_point("dropdown")

{'ok': True,
 'pose': (-0.1298930873527097, 1.0704577857291253, 1.5239824031203257)}

## Task 4 - `navigate_to_point` / `navigate_path`
`navigate_to_point` sends one `pose_nav` RPC. `navigate_path` deliberately **refuses** to fall back to sequential single-pose calls: pass a verified `native_path_callback` once one is confirmed for this SDK/firmware (see `academy/todo.txt`); until then this is a documented limitation, not a silent point-by-point substitute.

In [19]:
def navigate_to_point(point_name):
    x, y, yaw = _points[point_name]
    return slam_rpc.pose_nav(x, y, yaw)

def navigate_path(point_names, native_path_callback=None):
    if native_path_callback is None:
        raise RuntimeError(
            "No verified native complete-path SLAM API is configured; refusing a silent "
            "point-by-point fallback (see academy/todo.txt)."
        )
    return native_path_callback([_points[name] for name in point_names])

In [22]:
navigate_to_point("pickup")

(3104, None)

## Task 5 - `view_map()`: capture visualization *while mapping is running*
Subscribe to the live SLAM point-cloud topics and periodically save a 2-D scatter snapshot to disk during mapping; `view_map()` simply returns the newest saved snapshot. This sidesteps the "map is no longer accessible once saved on the mainboard" limitation noted in `notes.txt`.

In [23]:
import struct
import matplotlib.pyplot as plt
from unitree_sdk2py.idl.sensor_msgs.msg.dds_ import PointCloud2_

SLAM_POINT_TOPICS = [
    "rt/unitree/slam_mapping/points", "rt/unitree/slam_relocation/points",
    "rt/unitree/slam_relocation/global_map", "rt/unitree/slam_relocation/web_points",
]
cloud_subs = [(topic, Latest(topic, PointCloud2_)) for topic in SLAM_POINT_TOPICS]

def _decode_xy(msg, max_points=20000):
    fields = {f.name.lower(): f for f in msg.fields}
    x_off, y_off = fields["x"].offset, fields["y"].offset
    raw, step = bytes(msg.data), msg.point_step
    total = min(msg.width * msg.height, len(raw) // max(1, step))
    stride = max(1, total // max_points)
    xs, ys = [], []
    for i in range(0, total, stride):
        base = i * step
        xs.append(struct.unpack_from("<f", raw, base + x_off)[0])
        ys.append(struct.unpack_from("<f", raw, base + y_off)[0])
    return xs, ys

_map_snapshot_dir = Path("slam_map_snapshots"); _map_snapshot_dir.mkdir(exist_ok=True)

def capture_map_snapshot():
    candidates = [(t, s) for t, s in cloud_subs if s.message is not None]
    if not candidates:
        raise RuntimeError("No SLAM point-cloud message received yet; is mapping running?")
    topic, sub = max(candidates, key=lambda pair: pair[1].timestamp)
    xs, ys = _decode_xy(sub.message)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(xs, ys, s=0.5)
    ax.set_title(f"SLAM snapshot ({topic})"); ax.set_aspect("equal")
    path = _map_snapshot_dir / f"map_{int(time.time())}.png"
    fig.savefig(path, dpi=120); plt.close(fig)
    return path

def view_map():
    snapshots = sorted(_map_snapshot_dir.glob("map_*.png"))
    return snapshots[-1] if snapshots else None

capture_map_snapshot()  # call periodically while start_mapping() is active
view_map()

RuntimeError: No SLAM point-cloud message received yet; is mapping running?

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.